# Basic RAG Query Engine - Step by Step

[30-second quickstart](https://developers.llamaindex.ai/python/framework/#30-second-quickstart)

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG) application** using vector search, help you understand and implement a basic RAG system step by step:

**Install**

In [2]:
pip install langchain langchain-community langchain-chroma chromadb sentence-transformers python-dotenv 

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-6.3.0-py3-none-any.whl.metadata (20 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached pydantic_settings-2.16.3-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached onnxruntime-1.32.0-cp313-cp313-win_amd64.whl.metadata (6.1 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using 

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\patri\\AppData\\Roaming\\Python\\Python313\\site-packages\\onnxruntime\\transformers\\models\\longformer\\convert_to_onnx.py'
Check the permissions.



**Environment**

In [3]:
import os
from getpass import getpass
from dotenv import load_dotenv

from colorama import Fore, Style

# Load environment variables from .env file
load_dotenv(override=True)  

# Set OPENAI_API_KEY from environment variable
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded from environment variable.{Style.RESET_ALL}")

OpenAI API key loaded from environment variable.


**Initialize model**

In [4]:
from langchain.chat_models import init_chat_model


# -----------------------------
# LLM Setup
# -----------------------------

# A low-temperature model gives more repeatable demonstrations.
# explain
model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0,
    timeout=30,
    max_retries=3,
)

**Data Loading**

In [5]:
!mkdir -p 'data/forge/'
!wget \
  "https://raw.githubusercontent.com/magit/forge/refs/heads/main/CHANGELOG" \
  -O "data/forge/CHANGELOG.txt"

The syntax of the command is incorrect.
'wget' is not recognized as an internal or external command,
operable program or batch file.


**Document Loading**

In [7]:
%pip install langchain-community langchain-text-splitters

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(
    "data/forge/CHANGELOG.txt",
    encoding="utf-8",
)

documents = loader.load()

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
Using cached langchain_text_splitters-1.1.3-py3-none-any.whl (38 kB)

   ---------------------------------------- 0/3 [langchain-text-splitters]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   -------

C:\Users\patri\AppData\Local\Temp\ipykernel_24472\3988179235.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


**Documents Processing**

In [8]:

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(chunks[0].metadata)
print(documents[0].metadata)
print(documents[0].page_content[:300])



Documents loaded: 1
Chunks created: 96
{'source': 'data/forge/CHANGELOG.txt'}
{'source': 'data/forge/CHANGELOG.txt'}
# -*- mode: org -*-
* v0.6.7    2026-07-01

- Adjusted to breaking change in ~magit-branch-p~.  f3ad5910

- Fixed a regression when visiting blob or copying its URL.  #858

* v0.6.6    2026-06-01

- Fixed a regression in interaction with Ghub.  #848

- No longer error if ~ssh~ is missing when lookin


**Prompts**

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough




prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a general knowledge assistant.

Answer using only your existing knowledge.

Rules:
- Do not claim to have read local files.
- Do not invent details.
- If the answer depends on a specific document you cannot access,
  explicitly say that you do not have enough information.
- Keep the answer concise.""",
        ),
        ("human", "{question}"),
    ]
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a grounded knowledge assistant.

Use only the retrieved context to answer the question.

Rules:
- Cite supporting sources using their filenames in square brackets.
- Do not use outside knowledge.
- Do not invent facts.
- If the answer is not in the context, say so.
- Keep the answer focused on the question.""",
        ),
        (
            "human",
            """Retrieved context:

{context}

Question:
{question}""",
        ),
    ]
)


**utils**

In [10]:

def format_documents(docs: list[Document]) -> str:
    """Format retrieved documents with visible source labels."""

    if not docs:
        return "No relevant context was retrieved."

    formatted = []

    for position, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        year = doc.metadata.get("year", "unknown")
        topic = doc.metadata.get("topic", "unknown")

        formatted.append(
            f"[Source {position}]\n"
            f"File: {source}\n"
            f"Year: {year}\n"
            f"Topic: {topic}\n"
            f"Content: {doc.page_content}"
        )

    return "\n\n".join(formatted)

In [ ]:

basic_chain = (
    {
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [16]:

# build a basic RAG chain that retrieves context and uses it to answer questions
rag_chain = (
    {
        "context": retriever | RunnableLambda(lambda docs: format_documents(docs)),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

In [13]:
questions = [
    (
        "What pagination-related problem is described in the unreleased "
        "section of the Forge changelog?"
    ),
    (
        "What external platform changed the format of label IDs?"
    ),
    (
        "What operation could be completely prevented by the pagination bug?"
    ),
]

**Run without RAG**

In [14]:
answer = basic_chain.invoke({"question": "What pagination-related problem is described in the unreleased"})
print(f"{Fore.CYAN}\nANSWER WITHOUT RAG\n{Style.RESET_ALL}")
print(f"{Fore.MAGENTA}{answer}{Style.RESET_ALL}")

print(Style.RESET_ALL)


ANSWER WITHOUT RAG

I don't have enough information to determine what pagination-related problem is described in the unreleased document.



**Run with RAG**

In [17]:
answer = rag_chain.invoke("What pagination-related problem is described in the unreleased")
print(f"{Fore.CYAN}\nANSWER WITH RAG\n{Style.RESET_ALL}")
print(f"{Fore.GREEN}{answer}{Style.RESET_ALL}")

print(Style.RESET_ALL)


ANSWER WITH RAG

The pagination-related problem described is a bug that could completely prevent pulling API data for certain GitHub repositories. This bug was fixed in version 0.6.5 as noted in the changelog entry: 

"- Fixed a pagination bug, which could completely prevent pulling API data for certain Github repositories. #843" [Source 4]

